In [ ]:
# ============================================================
# GB_PI (Physics-Informed Gradient Boosting)
# Performance evaluation across 50 station-level splits
# Station-level 70:30 training-test split
#
# Author: Junyoung Lee
# Affiliation: Ulsan National Institute of Science and Technology (UNIST)
# Email: junyounglee@unist.ac.kr
# ============================================================

import numpy as np
import pandas as pd
import os
from pathlib import Path

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ------------------------------------------------------------
# Repository paths
# ------------------------------------------------------------
# This notebook can be run from either the repository root
# or the notebooks directory.
current_dir = Path.cwd()
project_dir = (
    current_dir.parent
    if current_dir.name == 'notebooks'
    else current_dir
)

data_dir = project_dir / 'data'
output_dir = project_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

input_file = data_dir / 'Total data_for submission.csv'
total_data = pd.read_csv(input_file)

# Use the absolute value of U.ratio
total_data['U.ratio'] = np.abs(total_data['U.ratio'])

station_col = 'SSN'
seeds = list(range(1, 51))

feature_sets = {
    'Set1': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle'],
    'Set2': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long'],
    'Set3': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'slope_500m'],
    'Set4': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long', 'slope_500m']
}

# -----------------------------
# Output folders
# -----------------------------
prediction_dir = os.path.join(output_dir, 'seed_predictions_GB_PI')
os.makedirs(prediction_dir, exist_ok=True)

# Directory for feature-importance results from individual seeds
importance_dir = os.path.join(output_dir, 'seed_feature_importance_GB_PI')
os.makedirs(importance_dir, exist_ok=True)

# -----------------------------
# Prepare PI target
# -----------------------------
total_data = total_data[
    (total_data['Vs30_mea'] > 0) &
    (total_data['Vs30_f0'] > 0)
].copy()

total_data['log_res'] = (
    np.log(total_data['Vs30_mea']) -
    np.log(total_data['Vs30_f0'])
)

all_results = []

# Store feature importance values for all seeds and feature sets
all_importances = []

# -----------------------------
# Repeat station-level split
# -----------------------------
for seed in seeds:

    np.random.seed(seed)

    stations = np.array(
        sorted(total_data[station_col].unique())
    )
    np.random.shuffle(stations)

    n_train = int(len(stations) * 0.7)

    train_stations = stations[:n_train]
    test_stations = stations[n_train:]

    data1 = total_data[
        total_data[station_col].isin(train_stations)
    ].copy()

    data2 = total_data[
        total_data[station_col].isin(test_stations)
    ].copy()

    y_test = data2['Vs30_mea'].values
    y_pred_p = data2['Vs30_f0'].values

    # -----------------------------
    # P-wave metrics
    # -----------------------------
    rmse_p = np.sqrt(
        mean_squared_error(y_test, y_pred_p)
    )
    r2_p = r2_score(y_test, y_pred_p)
    mae_p = mean_absolute_error(y_test, y_pred_p)
    bias_p = np.mean(y_pred_p - y_test)

    all_results.append({
        'seed': seed,
        'model': 'P-wave',
        'feature_set': 'P-wave',
        'RMSE': rmse_p,
        'R2': r2_p,
        'MAE': mae_p,
        'Bias': bias_p,
        'Delta_RMSE': 0.0,
        'Delta_R2': 0.0,
        'Delta_MAE': 0.0
    })

    pred_df = data2.copy()
    pred_df['P-wave_pred'] = y_pred_p

    # Store feature importance values for the current seed
    seed_importances = []

    # -----------------------------
    # GB_PI for Set1-Set4
    # -----------------------------
    for set_name, features in feature_sets.items():

        train_sub = data1[
            features + ['log_res']
        ].dropna()

        test_sub = data2[
            features
        ].dropna()

        idx = test_sub.index

        X_train = train_sub[features]
        y_train = train_sub['log_res']
        X_test = test_sub[features]

        model = GradientBoostingRegressor(
            n_estimators=500,
            learning_rate=0.01,
            max_depth=3,
            min_samples_split=5,
            min_samples_leaf=4,
            max_features=3,
            subsample=1,
            random_state=seed
        )

        model.fit(X_train, y_train)

        # -----------------------------
        # Variable importance
        # -----------------------------
        for feature, importance in zip(
            features,
            model.feature_importances_
        ):

            importance_result = {
                'seed': seed,
                'model': 'GB_PI',
                'feature_set': set_name,
                'feature': feature,
                'importance': importance
            }

            all_importances.append(importance_result)
            seed_importances.append(importance_result)

        # -----------------------------
        # Prediction
        # -----------------------------
        log_res_pred = model.predict(X_test)

        y_pred = (
            data2.loc[idx, 'Vs30_f0'].values *
            np.exp(log_res_pred)
        )

        y_true = data2.loc[idx, 'Vs30_mea'].values
        y_p_sub = data2.loc[idx, 'Vs30_f0'].values

        # -----------------------------
        # GB_PI metrics
        # -----------------------------
        rmse = np.sqrt(
            mean_squared_error(y_true, y_pred)
        )
        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        bias = np.mean(y_pred - y_true)

        # P-wave metrics on the same subset
        rmse_p_sub = np.sqrt(
            mean_squared_error(y_true, y_p_sub)
        )
        r2_p_sub = r2_score(y_true, y_p_sub)
        mae_p_sub = mean_absolute_error(
            y_true,
            y_p_sub
        )

        all_results.append({
            'seed': seed,
            'model': 'GB_PI',
            'feature_set': set_name,
            'RMSE': rmse,
            'R2': r2,
            'MAE': mae,
            'Bias': bias,
            'Delta_RMSE': rmse - rmse_p_sub,
            'Delta_R2': r2 - r2_p_sub,
            'Delta_MAE': mae - mae_p_sub
        })

        pred_df.loc[
            idx,
            f'{set_name}_pred'
        ] = y_pred

        # pred_df.loc[
        #     idx,
        #     f'GB_PI_{set_name}_log_res_pred'
        # ] = log_res_pred

    # -----------------------------
    # Save prediction by seed
    # -----------------------------
    pred_df.to_csv(
        os.path.join(
            prediction_dir,
            f'seed_{seed:02d}_predictions.csv'
        ),
        index=False
    )

    # -----------------------------
    # Save importance by seed
    # -----------------------------
    seed_importance_df = pd.DataFrame(
        seed_importances
    )

    seed_importance_df = seed_importance_df[
        [
            'seed',
            'model',
            'feature_set',
            'feature',
            'importance'
        ]
    ]

    seed_importance_df.to_csv(
        os.path.join(
            importance_dir,
            f'seed_{seed:02d}_feature_importance.csv'
        ),
        index=False
    )

# -----------------------------
# Save model results
# -----------------------------
results_df = pd.DataFrame(all_results)

results_df = results_df[
    [
        'seed',
        'model',
        'feature_set',
        'RMSE',
        'R2',
        'MAE',
        'Bias',
        'Delta_RMSE',
        'Delta_R2',
        'Delta_MAE'
    ]
]

results_df.to_csv(
    os.path.join(
        output_dir,
        '05-GB_PI_results.csv'
    ),
    index=False
)



In [ ]:
# -----------------------------
# Summary (mean)
# -----------------------------
summary_df = results_df.groupby(['model', 'feature_set'], sort=False).agg({
    'RMSE': 'mean',
    'R2': 'mean',
    'MAE': 'mean',
    'Bias': 'mean',
    'Delta_RMSE': 'mean',
    'Delta_R2': 'mean',
    'Delta_MAE': 'mean'
}).reset_index()

summary_df.to_csv(
    os.path.join(output_dir, '05-GB_PI_summary_mean.csv'),
    index=False
)

print(summary_df)

In [ ]:
# -----------------------------
# Save all variable importances
# -----------------------------
importance_df = pd.DataFrame(all_importances)

importance_df = importance_df[
    [
        'seed',
        'model',
        'feature_set',
        'feature',
        'importance'
    ]
]

importance_df.to_csv(
    os.path.join(
        output_dir,
        '07-GB_PI_feature_importance_by_seed.csv'
    ),
    index=False
)

# -----------------------------
# Summary across 50 seeds
# -----------------------------
importance_summary_df = (
    importance_df
    .groupby(
        [
            'model',
            'feature_set',
            'feature'
        ],
        as_index=False
    )
    .agg(
        importance_mean=('importance', 'mean'),
        importance_sd=('importance', 'std'),
        importance_median=('importance', 'median'),
        importance_min=('importance', 'min'),
        importance_max=('importance', 'max'),
        n_seed=('seed', 'nunique')
    )
)

importance_summary_df.to_csv(
    os.path.join(
        output_dir,
        '07-GB_PI_feature_importance_summary.csv'
    ),
    index=False
)

print(results_df.head(15))
print(importance_df.head(15))
print(importance_summary_df)